# chameleon: style cartridges for Qwen-Image

Give it a few images you like. It turns their **style** into a small *cartridge*: one code, about 1 KB. You can then
apply that style to new prompts or to your own photos with Qwen-Image 2.1 Turbo. It's roughly a home-made
Midjourney `--sref`.

How it works, in one line: a style encoder (MegaStyle) → a small describer head that writes the style out as text
(or as soft tokens) → Qwen-Image's text encoder → the image model. Nothing in the image model is fine-tuned.

The default head is **r7** (58.5M parameters). It was trained on 41k styles relabelled with style-only descriptions from a
small vision LLM, and its text scores about +0.10 CSD over the first head (r6c) against held-out Midjourney sref
images. r7 has no random-style prior yet: set `HEAD_REPO` to the r6c repo for `--sref random`-style codes.

**What to expect:** it carries palette, medium, lighting and mood well. It does not copy fine brushwork or grain the
way a real reference image would. *Texture boost* adds some of that back: it shows the model the reference
image itself, in its middle blocks only (12–19), which adds texture without copying the subject. This is a
research experiment, not a product.

**Runtime:** use a GPU with at least 22 GB, such as an L4 or A100 (*Runtime → Change runtime type*). A T4 won't fit.
The first run downloads about 40 GB of weights. After that, each image takes about 4 s on an A100 and about 20 s on an L4.

**Licences:** Qwen-Image 2.1 Turbo, FLUX.2-klein-base-4B (only its text-embedding matrix is used), the MegaStyle
encoder, and a describer head trained on MegaStyle-1.4M (research licence). Use it for research and personal
experiments, and use reference images you have the right to use.

In [ ]:
#@title 1 · setup (≈ 5 min)
REPO_URL = "https://github.com/kelechi-c/chameleon"  #@param {type:"string"}
HEAD_REPO = "tensorkelechi/chameleon-r7"  #@param ["tensorkelechi/chameleon-r7", "tensorkelechi/chameleon-r6c"] {allow-input: true}
import os, subprocess, sys
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"],
                     capture_output=True, text=True).stdout.strip()
print("gpu:", gpu or "none")
assert gpu and int(gpu.split(",")[-1]) >= 22000, "needs a gpu with >= 22 GB (L4 / A100): Runtime -> Change runtime type"
subprocess.run("pip uninstall -y -q torchao", shell=True)
subprocess.run('pip install -q "git+https://github.com/huggingface/diffusers@85c9fa4caee33d1c5b473395784471d2d7957c1b" '
               '"transformers==5.19.0" accelerate safetensors sentencepiece', shell=True, check=True)
if not os.path.isdir("/content/chameleon"):
    subprocess.run(["git", "clone", "-q", REPO_URL, "/content/chameleon"], check=True)
os.chdir("/content/chameleon")
sys.path[:0] = ["/content/chameleon/src", "/content/chameleon/scripts"]
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
print("ready")

In [ ]:
#@title 2 · describer head (and its random-style prior, if it has one)
from huggingface_hub import hf_hub_download, get_token
if get_token() is None:  # only needed while the repo is private: a read token with access, or set HF_TOKEN in colab secrets
    try:
        from huggingface_hub import notebook_login
        notebook_login()
    except Exception:
        pass
HEAD_NAME = HEAD_REPO.rsplit("-", 1)[-1]  # r7 / r6c: cartridges remember which head made them
try:
    HEAD = hf_hub_download(HEAD_REPO, "head_pre.pt")
except Exception as e:  # not on the hub: upload head_pre.pt (and code_prior.pt, if the head has one) by hand
    print(f"download from {HEAD_REPO} failed ({type(e).__name__}); upload head_pre.pt (+ code_prior.pt)")
    from google.colab import files
    for name, data in files.upload().items():
        open(f"/content/{name}", "wb").write(data)
    HEAD = "/content/head_pre.pt"
try:
    PRIOR = hf_hub_download(HEAD_REPO, "code_prior.pt")
except Exception:
    PRIOR = "/content/code_prior.pt" if os.path.exists("/content/code_prior.pt") else None
print(HEAD_NAME, HEAD, PRIOR or "no random-style prior for this head", sep="\n")

## your styles and photos

Upload one or more **style references**. Each image becomes its own cartridge. Tick `ONE_CARTRIDGE` to pool all the
uploads into a single style instead. **Photos to restyle** are optional. Leave the upload empty to use the bundled
bird and nature photos, which were generated for this project.

In [ ]:
#@title 3 · upload style references
ONE_CARTRIDGE = False  #@param {type:"boolean"}
import io, pathlib as pl
from PIL import Image
try:
    from google.colab import files
    up = files.upload()
except ImportError:  # local jupyter: put the images in ./my_refs
    up = {p.name: p.read_bytes() for p in sorted(pl.Path("my_refs").glob("*")) if p.suffix.lower() in (".jpg", ".jpeg", ".png", ".webp")}
imgs = {pl.Path(n).stem: Image.open(io.BytesIO(b)).convert("RGB") for n, b in up.items()}
assert imgs, "upload at least one style image"
REFS = {"mystyle": list(imgs.values())} if ONE_CARTRIDGE else {n: [im] for n, im in imgs.items()}
print({n: len(v) for n, v in REFS.items()})

In [ ]:
#@title 4 · photos to restyle (optional; skip the upload for the bundled ones)
USE_BUNDLED = True  #@param {type:"boolean"}
from chameleon.serve import square
PHOTOS = []
try:
    from google.colab import files
    up = {} if USE_BUNDLED else files.upload()
except ImportError:
    up = {}
PHOTOS = [square(Image.open(io.BytesIO(b)), 1024) for b in up.values()]
if not PHOTOS and USE_BUNDLED:
    PHOTOS = [square(Image.open(f"configs/nature/photo_{i}.jpg"), 1024) for i in (0, 2, 6)]
print(len(PHOTOS), "photos")

In [ ]:
#@title 5 · settings
PROMPTS = "a red fox in a snowy forest | a lighthouse on a rocky coast | a heron standing in a misty marsh"  #@param {type:"string"}
SEED = 7  #@param {type:"integer"}
RENDER = "text"  #@param ["text", "soft"]
TEXTURE_BOOST = True  #@param {type:"boolean"}
EDIT_PHOTOS = True  #@param {type:"boolean"}
EDIT_STRENGTH = 1.0  #@param {type:"slider", min:0.25, max:1.5, step:0.25}
MIX_FIRST_TWO = False  #@param {type:"boolean"}
INTERPOLATE_FIRST_TWO = False  #@param {type:"boolean"}
N_RANDOM = 2  #@param {type:"integer"}
RANDOM_SEED = 3  #@param {type:"integer"}
PROMPTS = [p.strip() for p in PROMPTS.split("|") if p.strip()]
print(PROMPTS)

**Settings:**
- `RENDER`: how the cartridge reaches the model. `text` (the describer's words) scored a little higher than `soft`
  (soft tokens) in our tests.
- `TEXTURE_BOOST`: adds a second image per prompt and per photo that also shows the model the reference image, in blocks 12–19 only. It gave
  +0.04–0.05 CSD with no extra copying of the reference. Random and mixed cartridges have no reference image, so they
  get no boost.
- `EDIT_STRENGTH`: values below 1 keep more of the original photo, and values above 1 push the style harder. It
  applies to the plain edit (the texture-boosted edit always runs at 1).
- `MIX_FIRST_TWO` / `INTERPOLATE_FIRST_TWO`: a 50/50 mix of the first two cartridges, or three steps between them
  (25 / 50 / 75 %). Mixed codes have no single reference image, so they get no texture boost.
- `N_RANDOM`: random styles sampled from the describer's prior, a bit like Midjourney's `--sref random`. Only heads
  with a prior (r6c for now) can do this.

In [ ]:
#@title 6 · load the models (once; ≈ 3 min after the first download)
import gc, torch
from chameleon import serve
pipe = serve.Pipeline(HEAD, PRIOR, head_name=HEAD_NAME)  # under 40 GB of gpu memory the vl encoder and the dit take turns
print("resident" if pipe.resident else "swapping encoder / dit per run")

In [ ]:
#@title 7 · run
OUT = pl.Path("/content/chameleon_out"); OUT.mkdir(exist_ok=True)
carts = {n: pipe.make(ims, [n] * len(ims)) for n, ims in REFS.items()}
if MIX_FIRST_TWO and len(carts) >= 2:
    a, b = list(carts)[:2]
    carts[f"{a}+{b}"] = pipe.mix([carts[a], carts[b]], [0.5, 0.5])
if INTERPOLATE_FIRST_TWO and len(carts) >= 2:
    a, b = [n for n in list(carts) if n in REFS][:2]
    for w in (0.25, 0.5, 0.75):
        carts[f"{a}→{b} {int(w * 100)}%"] = pipe.mix([carts[a], carts[b]], [1 - w, w])
if N_RANDOM and pipe.prior is None:
    print(f"head {HEAD_NAME} has no random-style prior: skipping N_RANDOM")
for k in range(N_RANDOM if pipe.prior is not None else 0):
    carts[f"random{RANDOM_SEED + k}"] = pipe.random(RANDOM_SEED + k)
photos = PHOTOS if EDIT_PHOTOS else []
results = {}
for n, c in carts.items():
    ref = REFS[n][0] if n in REFS else None  # texture boost needs a reference image
    said, out = pipe.run(c, PROMPTS, photos, ref, SEED, RENDER, TEXTURE_BOOST, EDIT_STRENGTH)
    c.describe = said
    c.save(OUT / f"{n}.cart")
    print(f"[{n}] {said}")
    results[n] = out
    for j, (label, im) in enumerate(out):
        im.save(OUT / f"{n}_{j:02d}.jpg", quality=95)
print(f"images and .cart files in {OUT}")

In [ ]:
#@title 8 · show: one grid per cartridge
from IPython.display import display
from PIL import ImageDraw

def grid(rows, cell=256):
    cols = max(len(r) for _, r in rows)
    im = Image.new("RGB", (cols * cell, len(rows) * (cell + 22)), "white")
    d = ImageDraw.Draw(im)
    for r, (label, row) in enumerate(rows):
        y = r * (cell + 22)
        d.text((6, y + 4), label, fill="black")
        for j, c in enumerate(row):
            if c is not None:
                im.paste(square(c, cell), (j * cell, y + 22))
    return im

for n, out in results.items():
    print(f"[{n}] {carts[n].describe}")
    rows = {}
    for label, im in out:  # "<prompt or photo k> · <t2i | t2i + texture | edit | edit + texture>"
        rows.setdefault(label.split(" · ")[0], []).append(im)
    ref = REFS[n][0] if n in REFS else None
    shown = []
    for k, (key, ims) in enumerate(rows.items()):
        lead = photos[int(key.split()[-1]) - 1] if key.startswith("photo ") else (ref if k == 0 else None)
        shown.append((key, [lead] + ims))
    display(grid(shown))

In [ ]:
#@title 9 · download everything (images + .cart files)
import shutil
shutil.make_archive("/content/chameleon_out", "zip", OUT)
try:
    from google.colab import files
    files.download("/content/chameleon_out.zip")
except ImportError:
    print("/content/chameleon_out.zip")

In [ ]:
#@title 10 · or: the web app (a public gradio link, valid ~72 h; reuses the loaded models)
os.environ["CHAMELEON_DEFER"] = "1"
sys.path.insert(0, "/content/chameleon/space")
import app
app.PIPE = pipe
app.demo.queue(max_size=8).launch(share=True, debug=False)

A `.cart` file is about 1 KB, so you can share it with anyone running this notebook and the same head (`r7` by
default).
`cg.Cartridge.load("x.cart")` gives you back the style. Cartridges from other heads won't work with this one.